# Ventas del 15 y 16 de septiembre

Para cada año, la venta del día se compara contra los **12 mismos días de la semana
anteriores** (si el 15 cayó miércoles, los 12 miércoles previos).

| Columna | Cálculo |
|---|---|
| Promedio 12 sem. | media de la venta de los 12 días de referencia |
| Desv. estándar | desviación estándar muestral de esos 12 días |
| Venta del día | venta del 15 (o 16) de septiembre |
| Dif. % | (venta del día ÷ promedio − 1) × 100 |
| Dif. σ | (venta del día − promedio) ÷ desviación estándar |
| Est. | estaciones incluidas |

**Estaciones incluidas.** Solo las que vendieron ese combustible **los 13 días** (los 12
de referencia y el día analizado). Un cierre nunca cuenta como venta 0: la estación sale
del cálculo de ese año. Si el día aún no está en la BD, se usan las estaciones completas en
la referencia y la venta del día queda vacía.

**Filtro de estaciones.** En el parámetro `ESTACIONES`: vacío = grupo completo; con una
o varias estaciones, se suman solo esas. Con el filtro, el Excel se guarda con el nombre de
las estaciones al final para no pisar el del grupo.

Celda teñida cuando |Dif. σ| ≥ 1 · azul = arriba del promedio · naranja = abajo.

## 0 · Parámetros

In [2]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

sys.path.insert(0, str(Path.cwd().parent / "codigo"))   # el paquete vive en <raíz>/codigo
from treher_analytics import data, config as cfg

FECHAS   = [(9, 15), (9, 16)]          # (mes, día)
SEMANAS  = 12
METRICA  = "litros"                    # "litros" o "pesos"
EXPORTAR = True

# Filtro de estaciones. Vacío = todo el grupo.
#   ESTACIONES = []                          -> grupo completo
#   ESTACIONES = ["IXTAZACUALA"]             -> una estación
#   ESTACIONES = ["QL", "TH", "AVE FENIX"]   -> varias (se suman)
ESTACIONES = ['AURORA']

COLS = {
    "litros": {"Total": "Total LTS", "Diésel": "Diésel (Lts)",
               "Premium": "Premium (Lts)", "Regular": "Regular (Lts)"},
    "pesos":  {"Total": "Total venta ($)", "Diésel": "Importe Diésel",
               "Premium": "Importe Premium", "Regular": "Importe Regular"},
}[METRICA]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
MESES = {9: "septiembre"}

## 1 · Datos

In [3]:
bd = data.cargar_bd()
F, E = cfg.COL_FECHA, cfg.COL_ESTACION
bd = bd[bd["Operó"]].copy()

# --- filtro de estaciones ---
DISPONIBLES = sorted(bd[E].unique())
sel = [str(s).strip().upper() for s in ESTACIONES]
desconocidas = [s for s in sel if s not in DISPONIBLES]
if desconocidas:
    raise ValueError(f"Estaciones no encontradas: {desconocidas}. "
                     f"Disponibles: {DISPONIBLES}")
if sel:
    bd = bd[bd[E].isin(sel)].copy()
    ALCANCE = ", ".join(sel)
else:
    ALCANCE = "Grupo completo"
SUFIJO = "" if not sel else "_" + "_".join(s.replace(" ", "") for s in sel)
SALIDA = Path.cwd() / f"Analisis_15_16_septiembre{SUFIJO}.xlsx"

ANIOS = sorted(bd[F].dt.year.unique())
print(f"alcance: {ALCANCE}")
print(f"BD: {bd[F].min():%d/%m/%Y} a {bd[F].max():%d/%m/%Y} · {len(bd):,} registros operados")
print(f"años: {ANIOS} · métrica: {METRICA}")
print(f"\nestaciones disponibles: {DISPONIBLES}")

alcance: AURORA
BD: 20/03/2021 a 13/09/2026 · 2,004 registros operados
años: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)] · métrica: litros

estaciones disponibles: ['ATB', 'ATEPO', 'AURORA', 'AVE FENIX', 'COMBULUB', 'HUEHUETOCA', 'IXTAZACUALA', 'QL', 'TEOLOYUCAN', 'TH', 'TOREMEX', 'TREHER LL', 'TREHER T']


## 2 · Cálculo

In [4]:
def analizar(anio, mes, dia, col):
    """Una fila de la tabla: año × combustible."""
    f = pd.Timestamp(anio, mes, dia)
    ref = [f - pd.Timedelta(weeks=k) for k in range(SEMANAS, 0, -1)]   # antiguo -> reciente
    if ref[0] < bd[F].min():
        return None

    w = bd[bd[F].isin(ref + [f])][[F, E, col]]
    w = w[w[col] > 0]                                  # NaN y 0 no son venta
    hay_dia = bool((w[F] == f).any())

    requeridas = ref + ([f] if hay_dia else [])
    cuenta = w[w[F].isin(requeridas)].groupby(E)[F].nunique()
    panel = sorted(cuenta[cuenta == len(requeridas)].index)
    wp = w[w[E].isin(panel)]

    # estaciones completas en la referencia que NO tienen el día cargado
    c_ref = w[w[F].isin(ref)].groupby(E)[F].nunique()
    completas_ref = set(c_ref[c_ref == len(ref)].index)
    faltan = sorted(completas_ref - set(panel)) if hay_dia else []

    serie = wp[wp[F].isin(ref)].groupby(F)[col].sum().reindex(ref)
    media = serie.mean()
    sd = serie.std(ddof=1)
    venta = wp.loc[wp[F] == f, col].sum() if hay_dia else np.nan

    fila = {
        "Año": anio,
        "Día": DIAS[f.dayofweek],
        "Est.": len(panel),
        "Promedio 12 sem.": media,
        "Desv. estándar": sd,
        "Venta del día": venta,
        "Dif. %": 100 * (venta / media - 1) if hay_dia else np.nan,
        "Dif. σ": (venta - media) / sd if hay_dia and sd else np.nan,
    }
    return fila, serie, panel, faltan


TABLAS, DETALLE, PANELES, AVISOS = {}, {}, {}, []
for mes, dia in FECHAS:
    filas, det, pan = [], {}, {}
    for comb, col in COLS.items():
        for a in ANIOS:
            r = analizar(a, mes, dia, col)
            if r is None:
                continue
            fila, serie, panel, faltan = r
            filas.append({"Combustible": comb, **fila})
            if comb == "Total":
                det[a], pan[a] = serie, panel
                if faltan:
                    AVISOS.append(f"{dia}-{MESES[mes][:3]}-{a}: sin venta del día cargada "
                                  f"para {', '.join(faltan)} -> fuera del cálculo")
    TABLAS[(mes, dia)] = pd.DataFrame(filas).set_index(["Combustible", "Año"])
    DETALLE[(mes, dia)], PANELES[(mes, dia)] = det, pan

print("tablas calculadas:", [f"{d} de {MESES[m]}" for m, d in TABLAS])
if AVISOS:
    print("\n⚠ Carga incompleta del día (esas estaciones no entran en su renglón):")
    for av in AVISOS:
        print("   ·", av)

tablas calculadas: ['15 de septiembre', '16 de septiembre']


In [5]:
FMT_V = "{:,.0f}" if METRICA == "litros" else "${:,.0f}"
AZUL, NARANJA = "#247BA0", "#D45F31"
F_AZUL, F_NAR = "#E8F1F5", "#FDECE4"


def estilo(t, titulo):
    def tinte(r):
        z = r["Dif. σ"]
        css = ""
        if pd.notna(z) and abs(z) >= 1:
            css = (f"background-color:{F_AZUL if z > 0 else F_NAR};"
                   f"color:{AZUL if z > 0 else NARANJA};font-weight:600")
        return [css if c in ("Dif. %", "Dif. σ") else "" for c in r.index]

    return (t.style
             .format({"Promedio 12 sem.": FMT_V, "Desv. estándar": FMT_V,
                      "Venta del día": FMT_V, "Dif. %": "{:+.1f}%",
                      "Dif. σ": "{:+.2f}", "Est.": "{:.0f}"}, na_rep="—")
             .apply(tinte, axis=1)
             .set_caption(titulo)
             .set_table_styles([
                 {"selector": "caption",
                  "props": "font-size:15px;font-weight:700;text-align:left;"
                           "color:#0B2244;padding-bottom:6px"},
                 {"selector": "th", "props": "text-align:right"},
                 {"selector": "th.row_heading.level0",
                  "props": "text-align:left;font-weight:700;color:#0B2244"},
                 {"selector": "td", "props": "padding:3px 10px"},
             ]))

## 3 · 15 de septiembre

In [6]:
display(estilo(TABLAS[(9, 15)], f"15 DE SEPTIEMBRE · {METRICA} · {ALCANCE}"))

## 4 · 16 de septiembre

In [7]:
display(estilo(TABLAS[(9, 16)], f"16 DE SEPTIEMBRE · {METRICA} · {ALCANCE}"))

## 5 · Detalle de los 12 días de referencia (Total)

Fecha y venta de cada día usado en el promedio, para auditar la tabla.

In [ ]:
for (mes, dia), det in DETALLE.items():
    tabla = pd.DataFrame(
        {a: [f"{fecha:%d/%m} · {FMT_V.format(v)}" if pd.notna(v) else f"{fecha:%d/%m} · —"
             for fecha, v in s.items()]
         for a, s in det.items()},
        index=[f"−{k} sem." for k in range(SEMANAS, 0, -1)])
    display(tabla.style.set_caption(f"Referencia del {dia} de {MESES[mes]} · Total")
                 .set_table_styles([{"selector": "caption",
                                     "props": "font-weight:700;text-align:left"}]))

## 6 · Estaciones incluidas por año (Total)

In [ ]:
todas = sorted(bd[E].unique())
for (mes, dia), pan in PANELES.items():
    m = pd.DataFrame({a: ["✓" if e in p else "" for e in todas] for a, p in pan.items()},
                     index=todas)
    m.loc["Total estaciones"] = [len(p) for p in pan.values()]
    display(m.style.set_caption(f"{dia} de {MESES[mes]}")
             .set_table_styles([{"selector": "caption",
                                 "props": "font-weight:700;text-align:left"}]))

## 7 · Exportar a Excel

In [ ]:
if EXPORTAR:
    redondeo = {"Promedio 12 sem.": 0, "Desv. estándar": 0, "Venta del día": 0,
                "Dif. %": 1, "Dif. σ": 2}
    try:
        with pd.ExcelWriter(SALIDA, engine="openpyxl") as xw:
            for (mes, dia), t in TABLAS.items():
                estilo(t.round(redondeo), "").to_excel(xw, sheet_name=f"{dia} sep")
        print("exportado:", SALIDA)
    except PermissionError:
        print("No se pudo exportar: el archivo está abierto en Excel. "
              "Ciérralo y vuelve a correr esta celda.")